## Setup

In [ ]:
%%capture
!pip install faiss-gpu google-cloud-bigquery google-cloud-storage \
             gcsfs pyarrow pandas torch --quiet

In [ ]:
!pip install faiss-cpu

In [ ]:
!git clone -b dev https://github.com/sbnikhil/RecoSys.git

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded")

fname = next(iter(uploaded.keys()))
path = os.path.abspath(fname)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = path
print("GOOGLE_APPLICATION_CREDENTIALS =", path)

In [ ]:
import sys
%cd /content/RecoSys
sys.path.insert(0, '/content/RecoSys')
!ls

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > A100 GPU")

In [ ]:
# Upload artifacts/500k/ from your local machine.
# Build the zip locally first:
#   cd artifacts && zip -r 500k.zip 500k/
# Then upload here.

from google.colab import files
import zipfile

print("Upload 500k.zip:")
uploaded = files.upload()
with zipfile.ZipFile("500k.zip", "r") as z:
    z.extractall("artifacts/")
print("Artifacts extracted:")
!ls artifacts/500k/

In [ ]:
import pathlib, json
assert pathlib.Path("artifacts/500k/vocabs.pkl").exists(), "Missing vocabs.pkl"
meta = json.loads(pathlib.Path("artifacts/500k/sequences_v2/metadata.json").read_text())
print(f"train_sessions: {meta['n_train_sessions']:,}")
print(f"val_sessions  : {meta['n_val_sessions']:,}")
print(f"n_items       : {meta['n_items']:,}")
print("Artifacts OK")

## SASRec V10 -- Canonical Redesign Sanity Check

Verify that the canonical SASRec redesign is in place before launching the long training run.

The redesign drops L2 normalisation from `forward()` and `get_item_embeddings()`, adds a learnable `log_scale` parameter, and switches the loss to gBCE with K=256 negatives. Three things to check:

1. **Trunc-normal init**: all transformer weight stds in [0.010, 0.025]. >= 0.05 means Xavier default is still active.
2. **Output norm != 1.0**: forward output norm should be > 1.0 (was exactly 1.0000 under L2). Exactly 1.0000 means `F.normalize` was not removed.
3. **`model.log_scale`**: must exist as an `nn.Parameter`, init `exp(0) = 1.0`. Missing means the new parameter wasn't added.

In [ ]:
import sys
sys.path.insert(0, '/content/RecoSys')
import torch
from src.sequence.models.sasrec import SASRecModel

m = SASRecModel(n_items=284524, embed_dim=128, n_layers=2, n_heads=4, ffn_dim=256)

# 1. Trunc-normal init
print("Encoder weight stds (all should be <= 0.025):")
any_bad = False
for name, param in m.encoder.named_parameters():
    if "weight" in name and param.dim() >= 2:
        std = param.std().item()
        flag = "  <-- FAIL" if std >= 0.05 else ""
        print(f"  {name:52s}  std={std:.4f}{flag}")
        if std >= 0.05:
            any_bad = True

# 2. Forward output should NOT be unit-norm any more
x = torch.randint(1, 284524, (2, 19))
e = torch.zeros_like(x)
out = m.forward(x, e)
last_norm = out[:, -1, :].norm(dim=-1).mean().item()
item_norm = m.get_item_embeddings()[1].norm().item()
print(f"\nforward last-position norm (mean): {last_norm:.4f}  (was 1.0000 under L2)")
print(f"item emb norm (sample item 1)    : {item_norm:.4f}  (was 1.0000 under L2)")
norm_bad = abs(last_norm - 1.0) < 1e-3 or abs(item_norm - 1.0) < 1e-3

# 3. Learnable log_scale must exist
has_scale = hasattr(m, "log_scale") and isinstance(m.log_scale, torch.nn.Parameter)
scale_val = m.log_scale.exp().item() if has_scale else float("nan")
print(f"\nlog_scale present: {has_scale}   exp(log_scale) at init: {scale_val:.4f}  (should be ~1.0)")

if any_bad:
    raise RuntimeError("Xavier init still present -- _init_weights fix was not applied.")
if norm_bad:
    raise RuntimeError("Output norm is 1.0 -- F.normalize was not removed from sasrec.py.")
if not has_scale:
    raise RuntimeError("model.log_scale missing -- the new learnable scalar was not added.")
print("\nAll three sanity checks PASSED.")
del m

## SASRec V10 Training (Session-Based, Canonical: raw IP + gBCE + learnable scale)

**Loss**: gBCE (gSASRec, RecSys 2023) with K=256 uniform random negatives, calibration parameter t=0.75.
**Scoring**: raw inner product (no L2 normalisation anywhere).
**Schedule**: per-step LR warmup (1000 steps) + cosine decay.

**Pass criterion (epoch 1, ~3 min on A100):**
- Train loss steadily decreasing (no plateau).
- `val NDCG@20 >= 0.05`.
- `val HR@20 > pop HR@20 (~0.077)` -- model must beat popularity by epoch 2.
- `log_scale.exp()` drifts upward to ~1.5-3.0 (printed each step at log_every=200).

If epoch-1 NDCG@20 < 0.02 or HR@20 < pop, the redesign did not land cleanly -- stop and re-diagnose.

**Convergence target**: best val NDCG@20 >= 0.22 (T4Rec GRU4Rec target; SASRec should match or beat).
Runtime: ~3-4 h on A100 to best checkpoint (typically epoch 12-18).

**batch-size note:** default 128. If OOM on a 40 GB A100, add `--batch-size 64`.

In [ ]:
!python scripts/sequence/train_sasrec_session.py
# A100 40GB: batch_size=128 (~10 GB peak VRAM)
# V100/T4:   add --batch-size 64

## Save SASRec V10 Artifacts to GCS

In [ ]:
import os, json, pathlib
from datetime import datetime, timezone
from google.cloud import storage

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/recosys-service-account.json"

BUCKET_NAME = "recosys-data-bucket"
GCS_PREFIX  = "models/sasrec_session_v10"

REPO_ROOT = pathlib.Path("/content/RecoSys")
CKPT_DIR  = REPO_ROOT / "artifacts/500k/sequences_v2/checkpoints_v10_sasrec_session"

log_path = CKPT_DIR / "training_log.json"
if log_path.exists():
    log  = json.loads(log_path.read_text())
    best = max(log, key=lambda r: r["val_ndcg_20"])
    tag  = f"ep{best['epoch']}_ndcg{best['val_ndcg_20']:.4f}"
else:
    tag  = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")

dest_prefix = f"{GCS_PREFIX}/{tag}"

client = storage.Client()
to_upload = list(CKPT_DIR.glob("*"))
print(f"Uploading {len(to_upload)} file(s) to gs://{BUCKET_NAME}/{dest_prefix}/\n")

for p in to_upload:
    blob_path = f"{dest_prefix}/{p.name}"
    bucket    = client.bucket(BUCKET_NAME)
    blob      = bucket.blob(blob_path)
    blob.upload_from_filename(str(p))
    print(f"  gs://{BUCKET_NAME}/{blob_path}  <=  {p.name}")

print(f"\nDone. Artifacts at: gs://{BUCKET_NAME}/{dest_prefix}/")